In [20]:
#DV: rating row[8]
import numpy as np
#import matplotlib.pyplot as plt
import csv,os
#from sklearn.linear_model import LogisticRegression
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import accuracy_score,precision_score, recall_score
#from sklearn.metrics import confusion_matrix
#from sklearn.inspection import permutation_importance
#import numpy as np
#from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier
#from tensorflow.keras.layers import Embedding
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

#import data from CSVs
data = []
files=["../2_Database/TopBeerData.csv","../2_Database/GoodBeerData.csv","../2_Database/RestBeerData.csv"]
for i in range (0,len(files)):
    with open(files[i],"r",encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # remove header
        for row in reader:
            data.append(row)

all_textual_data = []
textual_data = []
for row in data[1:]:  # ignore heading
    textual_data.append(
        f"brewery: {row[2]}\nbeer name: {row[3]} \nbeer kind: {row[4]} \ndescription: {row[11]}" # combined brewery beers name beer kind description
        f"\nABV: {row[6]} \nIBU: {row[7]} \nrating: {row[8]}" # combined ABV IBU rating
    )

# -------------------------
# MODEL
# -------------------------

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

embeddings = []
batch_size = 100
for i in range(0, len(textual_data), batch_size):
    batch = textual_data[i:i + batch_size]
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )
    embeddings.extend(
        item.embedding
        for item in response.data
    )

print(len(embeddings))

X_emb = np.array(embeddings)
print(X_emb.shape)

#get user beer preference
pref_beer = "Chef's Bottle"

for i, text in enumerate(textual_data):
    if pref_beer in text:
        print(i)
        print(text)
        break

print(X_emb[i])
id=i

#cosine similarity
pref = X_emb[id].reshape(1, -1)
similarities = cosine_similarity(pref, X_emb)[0]
top_k = 10
top_idx = np.argsort(similarities)[::-1]
top_idx = top_idx[top_idx != id][:10]
for i in top_idx:
    print(i, similarities[i], textual_data[i])


2302
(2302, 1536)
305
brewery: Mad Scientist
beer name: Chef's Bottle 
beer kind: Stout - Imperial / Double 
description: MAD SCIENTIST BARREL PROJECT team series Norbert Piszkor [Madhouse Budapest executive chef]Jamaican Rum Barrel Aged Imperial StoutCELLAR MASTER NOTESDo not eat. Taste. Savor. Relish.In general, our brewery doesn't embrace the mindset of "Simple is better" but sometimes it comes in that beautiful form we are just unable to ignore it. For this release, we used a freshly emptied Jamaican rum barrel and filled it up with our base imperial stout we've been improving for years now to provide the perfect backbone for our barrel project. The result is a chocolaty-spicy-somewhat fruity, full-bodied, yet not sweet, thick, black stout which never ceases to amaze the crew, anytime we pop a bottle. 
ABV: 10.6 
IBU: N/A 
rating: 3.99
[-0.00461197 -0.00402451 -0.01478577 ... -0.02279663  0.0082016
  0.00630188]
344 0.8407920935097393 brewery: Mad Scientist
beer name: Sweet Spot 
b